In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import jax
import jax.numpy as jnp

from model import create_model, SurfPINNConfig
from physics import compute_psnr, compute_rmse, mean_curvature_loss
from data_gen import load_dataset, generate_dataset

print(f"JAX devices: {jax.devices()}")

## 1. Load Data and Model

In [ ]:
# Generate data if not exists
import os
if not os.path.exists('../data/synthetic_dam_break.h5'):
    generate_dataset('../data/synthetic_dam_break.h5')

# Load dataset
data = load_dataset('../data/synthetic_dam_break.h5')

print("Dataset loaded:")
print(f"  Eulerian height: {data['eulerian']['height'].shape}")
print(f"  Lagrangian positions: {data['lagrangian']['positions'].shape}")

In [ ]:
# Initialize model (or load checkpoint)
import pickle

config = SurfPINNConfig(latent_dim=64)
model = create_model(config)
rng = jax.random.PRNGKey(42)

# Prepare input shapes
height = data['eulerian']['height']
coords = data['eulerian']['grid_coords']
positions = data['lagrangian']['positions']

nx, ny, nt = height.shape[:3]
n_particles = positions.shape[0]

# Sample at t=0
grid_input = np.concatenate([coords[:,:,0,:], height[:,:,0,:]], axis=-1)[None,...]
particle_pos = positions[:,0,:][None,...]

# Initialize or load
checkpoint_path = '../checkpoints/surfpinn_best.pkl'
if os.path.exists(checkpoint_path):
    with open(checkpoint_path, 'rb') as f:
        ckpt = pickle.load(f)
    params = jax.tree_util.tree_map(jnp.array, ckpt['params'])
    state = jax.tree_util.tree_map(jnp.array, ckpt['state'])
    print(f"Loaded checkpoint from epoch {ckpt['epoch']}")
else:
    params, state = model.init(rng, jnp.array(grid_input), jnp.array(particle_pos), True)
    print("Using randomly initialized model (no checkpoint found)")

## 2. Eulerian Height Field Visualization

In [ ]:
# Get predictions for multiple time steps
height_preds = []
height_trues = []

for t in range(nt):
    grid_t = np.concatenate([coords[:,:,t,:], height[:,:,t,:]], axis=-1)[None,...]
    part_t = positions[:,t,:][None,...]
    
    (h_pred, v_pred, _, _), _ = model.apply(
        params, state, None, 
        jnp.array(grid_t), jnp.array(part_t), 
        False
    )
    
    height_preds.append(np.array(h_pred[0,:,:,0]))
    height_trues.append(height[:,:,t,0])

print(f"Generated predictions for {nt} time steps")

In [ ]:
# Plot height fields at selected time steps
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

times = [0, nt//4, nt//2, nt-1]
for i, t in enumerate(times):
    # Ground truth
    im1 = axes[0, i].imshow(height_trues[t].T, origin='lower', cmap='viridis', vmin=0, vmax=1)
    axes[0, i].set_title(f'Ground Truth (t={t})')
    axes[0, i].set_xlabel('x')
    axes[0, i].set_ylabel('y')
    
    # Prediction
    im2 = axes[1, i].imshow(height_preds[t].T, origin='lower', cmap='viridis', vmin=0, vmax=1)
    axes[1, i].set_title(f'Prediction (t={t})')
    axes[1, i].set_xlabel('x')
    axes[1, i].set_ylabel('y')

plt.colorbar(im1, ax=axes[0,:], shrink=0.6, label='Height')
plt.colorbar(im2, ax=axes[1,:], shrink=0.6, label='Height')
plt.suptitle('SurfPINN: Eulerian Height Field Predictions', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# 3D surface plot
t = nt // 2  # Middle time step

x = np.linspace(0, 1, nx)
y = np.linspace(0, 1, ny)
X, Y = np.meshgrid(x, y, indexing='ij')

fig = plt.figure(figsize=(14, 5))

ax1 = fig.add_subplot(121, projection='3d')
ax1.plot_surface(X, Y, height_trues[t], cmap='viridis', alpha=0.8)
ax1.set_xlabel('X')
ax1.set_ylabel('Y')
ax1.set_zlabel('Height')
ax1.set_title('Ground Truth')
ax1.set_zlim(0, 1)

ax2 = fig.add_subplot(122, projection='3d')
ax2.plot_surface(X, Y, height_preds[t], cmap='viridis', alpha=0.8)
ax2.set_xlabel('X')
ax2.set_ylabel('Y')
ax2.set_zlabel('Height')
ax2.set_title('SurfPINN Prediction')
ax2.set_zlim(0, 1)

plt.suptitle(f'3D Height Field at t={t}', fontsize=14)
plt.tight_layout()
plt.show()

## 3. Lagrangian Particle Trajectories

In [ ]:
# Plot particle trajectories
n_plot = min(50, n_particles)  # Plot subset for clarity

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# XZ plane (side view)
for i in range(n_plot):
    traj = positions[i, :, :]  # (nt, 3)
    axes[0].plot(traj[:, 0], traj[:, 2], alpha=0.5, linewidth=0.5)

axes[0].scatter(positions[:n_plot, 0, 0], positions[:n_plot, 0, 2], 
               c='green', s=10, label='Start', zorder=5)
axes[0].scatter(positions[:n_plot, -1, 0], positions[:n_plot, -1, 2], 
               c='red', s=10, label='End', zorder=5)
axes[0].set_xlabel('X')
axes[0].set_ylabel('Z (Height)')
axes[0].set_title('Particle Trajectories (XZ Side View)')
axes[0].legend()
axes[0].set_xlim(0, 1)

# XY plane (top view)
for i in range(n_plot):
    traj = positions[i, :, :]
    axes[1].plot(traj[:, 0], traj[:, 1], alpha=0.5, linewidth=0.5)

axes[1].scatter(positions[:n_plot, 0, 0], positions[:n_plot, 0, 1], 
               c='green', s=10, label='Start', zorder=5)
axes[1].scatter(positions[:n_plot, -1, 0], positions[:n_plot, -1, 1], 
               c='red', s=10, label='End', zorder=5)
axes[1].set_xlabel('X')
axes[1].set_ylabel('Y')
axes[1].set_title('Particle Trajectories (XY Top View)')
axes[1].legend()
axes[1].set_xlim(0, 1)
axes[1].set_ylim(0, 1)

plt.suptitle('Lagrangian Surface Particle Paths', fontsize=14)
plt.tight_layout()
plt.show()

## 4. Evaluation Metrics

In [ ]:
# Compute metrics for all time steps
psnrs = []
rmses = []

for t in range(nt):
    pred = jnp.array(height_preds[t])
    true = jnp.array(height_trues[t])
    
    psnrs.append(float(compute_psnr(pred, true)))
    rmses.append(float(compute_rmse(pred, true)))

print("Metrics Summary:")
print(f"  PSNR: {np.mean(psnrs):.2f} ± {np.std(psnrs):.2f} dB")
print(f"  RMSE: {np.mean(rmses):.6f} ± {np.std(rmses):.6f}")
print(f"  Min PSNR: {np.min(psnrs):.2f} dB at t={np.argmin(psnrs)}")
print(f"  Max PSNR: {np.max(psnrs):.2f} dB at t={np.argmax(psnrs)}")

In [ ]:
# Plot metrics over time
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(range(nt), psnrs, 'b-o', markersize=4)
axes[0].axhline(np.mean(psnrs), color='r', linestyle='--', label=f'Mean: {np.mean(psnrs):.1f} dB')
axes[0].set_xlabel('Time Step')
axes[0].set_ylabel('PSNR (dB)')
axes[0].set_title('PSNR Over Time')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(range(nt), rmses, 'g-o', markersize=4)
axes[1].axhline(np.mean(rmses), color='r', linestyle='--', label=f'Mean: {np.mean(rmses):.4f}')
axes[1].set_xlabel('Time Step')
axes[1].set_ylabel('RMSE')
axes[1].set_title('RMSE Over Time')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Curvature Analysis

In [ ]:
# Compute curvature for ground truth and predictions
def compute_curvature_map(h, dx=1.0, dy=1.0):
    """Compute mean curvature at each point."""
    h = np.array(h)
    
    # Gradients
    h_x = np.gradient(h, dx, axis=0)
    h_y = np.gradient(h, dy, axis=1)
    h_xx = np.gradient(h_x, dx, axis=0)
    h_yy = np.gradient(h_y, dy, axis=1)
    h_xy = np.gradient(h_x, dy, axis=1)
    
    # Mean curvature
    grad_sq = 1 + h_x**2 + h_y**2
    num = (1 + h_y**2)*h_xx - 2*h_x*h_y*h_xy + (1 + h_x**2)*h_yy
    kappa = num / (grad_sq**1.5 + 1e-8)
    
    return kappa

t = nt // 2
dx = 1.0 / nx
dy = 1.0 / ny

curv_true = compute_curvature_map(height_trues[t], dx, dy)
curv_pred = compute_curvature_map(height_preds[t], dx, dy)

In [ ]:
# Plot curvature maps
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

vmax = max(np.abs(curv_true).max(), np.abs(curv_pred).max())

im1 = axes[0].imshow(curv_true.T, origin='lower', cmap='RdBu_r', vmin=-vmax, vmax=vmax)
axes[0].set_title('Ground Truth Curvature')
plt.colorbar(im1, ax=axes[0])

im2 = axes[1].imshow(curv_pred.T, origin='lower', cmap='RdBu_r', vmin=-vmax, vmax=vmax)
axes[1].set_title('Prediction Curvature')
plt.colorbar(im2, ax=axes[1])

# Curvature histogram
axes[2].hist(curv_true.flatten(), bins=50, alpha=0.5, label='Ground Truth', density=True)
axes[2].hist(curv_pred.flatten(), bins=50, alpha=0.5, label='Prediction', density=True)
axes[2].set_xlabel('Curvature')
axes[2].set_ylabel('Density')
axes[2].set_title('Curvature Distribution')
axes[2].legend()

plt.suptitle(f'Mean Curvature Analysis (t={t})', fontsize=14)
plt.tight_layout()
plt.show()

print(f"Curvature RMS - Ground Truth: {np.sqrt(np.mean(curv_true**2)):.4f}")
print(f"Curvature RMS - Prediction: {np.sqrt(np.mean(curv_pred**2)):.4f}")

## Summary

This notebook demonstrates:
1. Loading SurfPINN model and synthetic dam-break data
2. Visualizing Eulerian height fields (2D heatmaps and 3D surfaces)
3. Plotting Lagrangian particle trajectories
4. Computing PSNR and RMSE metrics
5. Analyzing surface curvature distribution

Key metrics to track:
- PSNR > 20 dB indicates good height field reconstruction
- Low curvature RMS suggests smooth, physically plausible surfaces